In [2]:
from pathlib import Path
import duckdb

current_folder = Path.cwd()

project_root = (
    current_folder.parent
    if current_folder.name == "notebooks"
    else current_folder
)

processed_data_path = project_root / "data" / "processed"

print(f"Project folder: {project_root}")
print(f"Processed data folder exists: {processed_data_path.is_dir()}")
print(f"Parquet files found: {len(list(processed_data_path.glob('*.parquet')))}")

Project folder: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform
Processed data folder exists: True
Parquet files found: 9


In [3]:
database_folder = project_root / "data" / "warehouse"
database_folder.mkdir(parents=True, exist_ok=True)

database_path = database_folder / "customer_loyalty.duckdb"

con = duckdb.connect(str(database_path))

print(f"Connected to: {database_path}")

Connected to: /Users/faridoonfarahi/Documents/Personal_Projects/customer_loyalty_platform/data/warehouse/customer_loyalty.duckdb


In [4]:
product_file = str(
    processed_data_path / "product_clean.parquet"
).replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE VIEW products AS
    SELECT *
    FROM read_parquet('{product_file}')
""")

con.sql("""
    SELECT PRODUCT_ID, DEPARTMENT, COMMODITY_DESC
    FROM products
    ORDER BY PRODUCT_ID
    LIMIT 5
""").df()

,PRODUCT_ID,DEPARTMENT,COMMODITY_DESC
0,25671,GROCERY,FRZN ICE
1,26081,MISC. TRANS.,NaN
2,26093,PASTRY,BREAD
3,26190,GROCERY,FRUIT - SHELF STABLE
4,26355,GROCERY,COOKIES/CONES


In [5]:
transaction_file = str(
    processed_data_path / "transaction_data.parquet"
).replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE VIEW transactions AS
    SELECT *
    FROM read_parquet('{transaction_file}')
""")

con.sql("""
    SELECT
        COUNT(*) AS transaction_rows,
        COUNT(DISTINCT household_key) AS households,
        COUNT(DISTINCT BASKET_ID) AS baskets
    FROM transactions
""").df()

,transaction_rows,households,baskets
0,2595732,2500,276484


In [6]:
con.sql("""
    SELECT
        t.household_key,
        t.BASKET_ID,
        t.PRODUCT_ID,
        p.DEPARTMENT,
        p.COMMODITY_DESC,
        t.QUANTITY,
        t.SALES_VALUE
    FROM transactions AS t
    LEFT JOIN products AS p
        ON t.PRODUCT_ID = p.PRODUCT_ID
    ORDER BY t.DAY, t.BASKET_ID, t.PRODUCT_ID
    LIMIT 5
""").df()

,household_key,BASKET_ID,PRODUCT_ID,DEPARTMENT,COMMODITY_DESC,QUANTITY,SALES_VALUE
0,2375,26984851472,1004906,PRODUCE,POTATOES,1,1.39
1,2375,26984851472,1033142,PRODUCE,ONIONS,1,0.82
2,2375,26984851472,1036325,PRODUCE,VEGETABLES - ALL OTHERS,1,0.99
3,2375,26984851472,1082185,PRODUCE,TROPICAL FRUIT,1,1.21
4,2375,26984851472,8160430,PRODUCE,ORGANICS FRUIT & VEGETABLES,1,1.50


In [7]:
con.sql("""
    SELECT
        COUNT(*) AS rows_after_join,
        COUNT(*) FILTER (
            WHERE p.PRODUCT_ID IS NULL
        ) AS unmatched_product_rows
    FROM transactions AS t
    LEFT JOIN products AS p
        ON t.PRODUCT_ID = p.PRODUCT_ID
""").df()

,rows_after_join,unmatched_product_rows
0,2595732,0


In [8]:
campaign_file = str(
    processed_data_path / "campaign_desc.parquet"
).replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE VIEW campaigns AS
    SELECT *
    FROM read_parquet('{campaign_file}')
""")

con.sql("""
    SELECT CAMPAIGN, DESCRIPTION, START_DAY, END_DAY
    FROM campaigns
    ORDER BY CAMPAIGN
    LIMIT 5
""").df()

,CAMPAIGN,DESCRIPTION,START_DAY,END_DAY
0,1,TypeB,346,383
1,2,TypeB,351,383
2,3,TypeC,356,412
3,4,TypeB,372,404
4,5,TypeB,377,411


In [9]:
targeting_file = str(
    processed_data_path / "campaign_table.parquet"
).replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE VIEW campaign_targets AS
    SELECT *
    FROM read_parquet('{targeting_file}')
""")

con.sql("""
    SELECT
        COUNT(*) AS targeting_records,
        COUNT(DISTINCT household_key) AS targeted_households,
        COUNT(DISTINCT CAMPAIGN) AS campaigns
    FROM campaign_targets
""").df()

,targeting_records,targeted_households,campaigns
0,7208,1584,30


In [10]:
con.sql("""
    SELECT
        ct.household_key,
        ct.CAMPAIGN,
        c.DESCRIPTION,
        c.START_DAY,
        c.END_DAY
    FROM campaign_targets AS ct
    LEFT JOIN campaigns AS c
        ON ct.CAMPAIGN = c.CAMPAIGN
    ORDER BY ct.CAMPAIGN, ct.household_key
    LIMIT 5
""").df()

,household_key,CAMPAIGN,DESCRIPTION,START_DAY,END_DAY
0,68,1,TypeB,346,383
1,93,1,TypeB,346,383
2,105,1,TypeB,346,383
3,293,1,TypeB,346,383
4,529,1,TypeB,346,383


In [11]:
con.execute("""
    CREATE OR REPLACE VIEW household_campaigns AS
    SELECT
        ct.household_key,
        ct.CAMPAIGN,
        c.DESCRIPTION,
        c.START_DAY,
        c.END_DAY
    FROM campaign_targets AS ct
    LEFT JOIN campaigns AS c
        ON ct.CAMPAIGN = c.CAMPAIGN
""")

con.sql("""
    SELECT
        COUNT(*) AS household_campaign_pairs,
        COUNT(*) FILTER (
            WHERE START_DAY IS NULL OR END_DAY IS NULL
        ) AS pairs_missing_dates
    FROM household_campaigns
""").df()

,household_campaign_pairs,pairs_missing_dates
0,7208,0


In [12]:
redemption_file = str(
    processed_data_path / "coupon_redempt.parquet"
).replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE VIEW coupon_redemptions AS
    SELECT *
    FROM read_parquet('{redemption_file}')
""")

con.sql("""
    SELECT
        COUNT(*) AS redemption_records,
        COUNT(DISTINCT household_key) AS redeeming_households,
        COUNT(DISTINCT CAMPAIGN) AS campaigns
    FROM coupon_redemptions
""").df()

,redemption_records,redeeming_households,campaigns
0,2318,434,30


In [13]:
coupon_file = str(
    processed_data_path / "coupon_clean.parquet"
).replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE VIEW coupon_eligibility AS
    SELECT *
    FROM read_parquet('{coupon_file}')
""")

con.sql("""
    SELECT
        COUNT(*) AS eligibility_records,
        COUNT(DISTINCT COUPON_UPC) AS coupon_identifiers,
        COUNT(DISTINCT CAMPAIGN) AS campaigns
    FROM coupon_eligibility
""").df()

,eligibility_records,coupon_identifiers,campaigns
0,119384,1135,30


In [14]:
demographic_file = str(
    processed_data_path / "hh_demographic.parquet"
).replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE VIEW household_demographics AS
    SELECT *
    FROM read_parquet('{demographic_file}')
""")

con.sql("""
    SELECT
        COUNT(*) AS demographic_records,
        COUNT(DISTINCT household_key) AS households
    FROM household_demographics
""").df()

,demographic_records,households
0,801,801


In [15]:
promotion_file = str(
    processed_data_path / "promotion_summary.parquet"
).replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE VIEW promotions AS
    SELECT *
    FROM read_parquet('{promotion_file}')
""")

con.sql("""
    SELECT
        COUNT(*) AS product_store_week_records,
        COUNT(*) FILTER (
            WHERE display_conflict
        ) AS display_conflict_records
    FROM promotions
""").df()

,product_store_week_records,display_conflict_records
0,36771279,15208


In [16]:
con.sql("""
    SELECT table_name
    FROM information_schema.views
    WHERE table_catalog = current_database()
      AND table_schema = 'main'
    ORDER BY table_name
""").df()

,table_name
0,campaign_targets
1,campaigns
2,coupon_eligibility
3,coupon_redemptions
4,household_campaigns
5,household_demographics
6,products
7,promotions
8,transactions


In [17]:
con.close()

con = duckdb.connect(str(database_path))

con.sql("""
    SELECT COUNT(*) AS household_campaign_pairs
    FROM household_campaigns
""").df()

,household_campaign_pairs
0,7208


In [18]:
con.close()
print("Warehouse connection closed.")

Warehouse connection closed.
